### 1. Import Libraries
Load spatial libraries to run the Voronoi generation and mapping.

In [1]:
import pandas as pd
import geopandas as gpd
from shapely.ops import voronoi_diagram
from shapely.geometry import MultiPoint
import io
import warnings

warnings.filterwarnings('ignore')

### 2. Load the Live Data from Google Sheets
This pulls the dataset containing all 362 teams. You can update `Furthest_Round` dynamically in your Google Sheet, and immediately rerun this notebook to see the updated maps!

In [2]:
# IMPORTANT: Replace this placeholder ID with your newly created Google Sheet's ID.
# Ensure the Google Sheet is set to "Anyone with the link can view" -> Viewer.

sheet_id = "1dZO435DeLvW9RfHgLL_eko2gfbK4MPUR669L5qBxFkU"
sheet_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv"
print(f"Downloading active teams from Google Sheets...")

try:
    all_teams_df = pd.read_csv(sheet_url)
    print(f"Loaded {len(all_teams_df)} teams from your live Google Sheet!")
except Exception as e:
    print(f"Error loading Google Sheet. Did you set `Anyone with the link can view`?\nError: {e}")
    print("\nFalling back to local generated CSV (`ncaa_d1_teams_initial.csv`)...")
    try:
        all_teams_df = pd.read_csv("ncaa_d1_teams_initial.csv")
        print(f"Loaded {len(all_teams_df)} teams from local fallback CSV instead.")
    except FileNotFoundError:
        print("Local CSV also not found. Please run the `scraper.py` script first to generate the dataset!")

Loaded 365 teams from your live Google Sheet!


### 3. Load the US Boundary
Read a US States GeoJSON directly from a web URL, dissolve the state borders into a single national polygon, and clip it to the contiguous lower 48 states.

In [3]:
print("Fetching 50-State US Boundary from the web...")

full_us_epsg = 3857 

url = "https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json"
states = gpd.read_file(url)

states = states[(states['id'] != '72') & (states['id'] != '02')]
us_full = states.dissolve()
us_boundary = us_full.to_crs(epsg=full_us_epsg).buffer(1500) # Buffer 1.5km to protect coastal islands

print("50-State Boundary ready!")

Fetching 50-State US Boundary from the web...
50-State Boundary ready!


### 4. The Smart Voronoi GeoJSON Generator
This dynamically draws the territories, clips them cleanly to the US boundary, and exports a lightweight map file for any active iteration round. It now embeds ALL column data into the features.

In [4]:
import math

def export_round_geojson(target_round):
    print(f"Processing Round of {target_round}...")
    
    active_teams = all_teams_df[all_teams_df['Furthest_Round'] <= target_round].copy()
    if active_teams.empty:
        print(f"Skipping Round {target_round} - no active teams found.")
        return
        
    active_teams.reset_index(drop=True, inplace=True)
    
    teams_gdf = gpd.GeoDataFrame(
        active_teams, 
        geometry=gpd.points_from_xy(active_teams.Lon, active_teams.Lat), 
        crs="EPSG:4326"
    ).to_crs(epsg=full_us_epsg)

    # Handle edge case where there is only 1 team left (the champion won't make a valid Voronoi)
    if len(teams_gdf) == 1:
        final_territories = us_boundary.copy()
        for col in active_teams.columns:
            final_territories[col] = active_teams[col].iloc[0]
    else:
        team_points = MultiPoint(teams_gdf.geometry.values)
        voronoi_polys = voronoi_diagram(team_points, envelope=us_boundary.unary_union.envelope)
        voronoi_gdf = gpd.GeoDataFrame(geometry=list(voronoi_polys.geoms), crs=full_us_epsg)

        # Pass ALL loaded Google sheet columns into the map file dynamically
        territories = gpd.sjoin(
            voronoi_gdf, 
            teams_gdf, 
            how='inner', 
            predicate='intersects'
        )
        final_territories = gpd.clip(territories, us_boundary)
        
    final_territories = final_territories.reset_index(drop=True)
    # Note: Simplify logic smoothed border lines to lower file size, but some gaps can occur.
    final_territories['geometry'] = final_territories['geometry'].simplify(500)
    final_territories = final_territories.to_crs(epsg=4326)
    
    filename = f"round_{target_round}.geojson"
    final_territories.to_file(filename, driver="GeoJSON")
    print(f"Saved {filename} with 50 states!")

### 5. Execute Pipeline
Automatically generate geojson maps for all rounds currently detected in your Google Sheet.

In [5]:
# Find all unique rounds currently present in the dataset (e.g. 362, 68, 64...)
try:
    rounds_to_generate = sorted([int(r) for r in all_teams_df['Furthest_Round'].dropna().unique()], reverse=True)
    
    for r in rounds_to_generate:
        export_round_geojson(r)
        
    print("\nAll perfect Voronoi maps generated successfully based on your Google Sheet data!")
except NameError:
    print("Error: all_teams_df not found. Please run cell 2 above to load your data.")

Processing Round of 362...
Saved round_362.geojson with 50 states!
Processing Round of 68...
Saved round_68.geojson with 50 states!
Processing Round of 64...
Saved round_64.geojson with 50 states!
Processing Round of 32...
Saved round_32.geojson with 50 states!
Processing Round of 16...
Saved round_16.geojson with 50 states!
Processing Round of 8...
Saved round_8.geojson with 50 states!
Processing Round of 4...
Saved round_4.geojson with 50 states!
Processing Round of 2...
Saved round_2.geojson with 50 states!

All perfect Voronoi maps generated successfully based on your Google Sheet data!


### 6. Interactive Preview using Dual-Layer Folium
Render the generated map for the most recent round containing two togglable layers: `Team Territories` (Polygons) and `Arena Locations` (Points).

In [6]:
import folium
import json

try:
    m = folium.Map(location=[39.8283, -98.5795], zoom_start=4, tiles='CartoDB positron')

    # Show the most recent round automatically
    current_round = int(all_teams_df['Furthest_Round'].min())
    
    # Load Polygon Layer Data
    with open(f'round_{current_round}.geojson', 'r') as f:
        geojson_data = json.load(f)

    # Dynamically build tooltips picking up optional Google sheet fields (Conference, Seed, Region, Best_Finish)
    poly_fields = ['School', 'Mascot']
    poly_aliases = ['School:', 'Mascot:']
    
    if len(geojson_data['features']) > 0:
        first_prop = geojson_data['features'][0]['properties']
        for col, alias in [('Conference', 'Conference:'), ('Seed', 'Seed:'), 
                           ('Region', 'Region:'), ('Best_Finish', 'Best Finish:')]:
            if col in first_prop and pd.notna(first_prop[col]):
                poly_fields.append(col)
                poly_aliases.append(alias)

    # Add Layer 1: Territories
    polygon_group = folium.FeatureGroup(name="Team Territories")
    folium.GeoJson(
        geojson_data,
        style_function=lambda feature: {
            'fillColor': feature['properties'].get('Hex', '#CCCCCC'),
            'fillOpacity': 0.70,
            'color': '#ffffff', 
            'weight': 1.5
        },
        tooltip=folium.GeoJsonTooltip(
            fields=poly_fields,
            aliases=poly_aliases,
            localize=True,
            style=("background-color: white; color: #333333; font-family: arial; font-size: 14px; padding: 10px;")
        )
    ).add_to(polygon_group)
    polygon_group.add_to(m)

    # Add Layer 2: Arenas
    arena_group = folium.FeatureGroup(name="Arena Locations")
    active_teams = all_teams_df[all_teams_df['Furthest_Round'] <= current_round]
    
    for _, row in active_teams.iterrows():
        arena_name = row.get('Arena', 'Unknown Arena')
        capacity = row.get('Capacity', 'N/A')
        attendance = row.get('Avg_Attendance', 'N/A')
        c1 = str(row.get('Hex', '#FFFFFF')).strip()
        c2 = str(row.get('Hex2', '#000000')).strip()
        
        # Invert Hex styling so the dot contrasts against the territory
        # Using Hex2 (secondary color) for fill, and Hex (primary) for ring
        
        # Construct the HTML tooltip popup manually for dots
        popup_html = f"<div style='font-family:arial; font-size: 14px; min-width: 150px;'>"
        popup_html += f"<b>{arena_name}</b><br><hr style='margin: 4px 0'>"
        popup_html += f"Max Capacity: {capacity}<br>Avg Attendance: {attendance}</div>"
        
        folium.CircleMarker(
            location=[row['Lat'], row['Lon']],
            radius=5,
            color='#ffffff',      # clean white outer line
            weight=1.5,
            fill=True,
            fill_color=c2,        # Secondary color fill
            fill_opacity=1,
            tooltip=folium.Tooltip(popup_html)
        ).add_to(arena_group)
        
    arena_group.add_to(m)

    # Add toggle control to the map
    folium.LayerControl(position='topright').add_to(m)

    display(m)
except Exception as e:
    print(f"Could not preview map: {e}\nMake sure the data loaded successfully first.")